## **AI Agents Course**

In [2]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# The Complete Guide to Building Your First AI Agent with DeepSeek\n",
    "\n",
    "This notebook adapts the agent from the Medium article to use the DeepSeek LLM instead of OpenAI. We will build an agent that can:\n",
    "1.  Search the web using DuckDuckGo.\n",
    "2.  Get live stock prices using Yahoo Finance."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Step 1: Install Required Libraries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%pip install -q langchain langchain-deepseek duckduckgo-search yfinance"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Step 2: Imports and API Key Setup"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "from langchain.agents import AgentExecutor, create_tool_calling_agent\n",
    "from langchain_core.prompts import ChatPromptTemplate\n",
    "from langchain.tools import tool, DuckDuckGoSearchRun\n",
    "\n",
    "# <<< MODIFIED: Import ChatDeepseek instead of ChatOpenAI\n",
    "from langchain_deepseek import ChatDeepseek "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- API Key Configuration ---\n",
    "# For security, it's best to set this as an environment variable.\n",
    "# However, for this notebook, you can paste it directly.\n",
    "\n",
    "# os.environ[\"DEEPSEEK_API_KEY\"] = \"YOUR_DEEPSEEK_API_KEY\"\n",
    "\n",
    "# <<< ⚠️ PASTE YOUR DEEPSEEK API KEY HERE\n",
    "DEEPSEEK_API_KEY = \"YOUR_DEEPSEEK_API_KEY\""
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Step 3: Define the Agent's Tools\n",
    "\n",
    "Tools are the functions the agent can decide to call. We'll create two:\n",
    "1. A pre-built tool for web search.\n",
    "2. A custom tool for getting stock prices."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Tool 1: Web Search (pre-built)\n",
    "search_tool = DuckDuckGoSearchRun()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Tool 2: Get Stock Price (custom)\n",
    "@tool\n",
    "def get_stock_price(symbol: str) -> float:\n",
    "    \"\"\"Returns the latest stock price for a given ticker symbol.\"\"\"\n",
    "    import yfinance as yf\n",
    "    try:\n",
    "        ticker = yf.Ticker(symbol)\n",
    "        price = ticker.info.get('regularMarketPrice')\n",
    "        if price is not None:\n",
    "            return price\n",
    "        else:\n",
    "            # Fallback for pre-market/post-market or if regularMarketPrice is missing\n",
    "            hist = ticker.history(period=\"1d\")\n",
    "            if not hist.empty:\n",
    "                return hist['Close'].iloc[-1]\n",
    "            return f\"Price for symbol {symbol} not found.\"\n",
    "    except Exception as e:\n",
    "        return f\"An error occurred for symbol {symbol}: {e}\""
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a list of all tools the agent can use\n",
    "tools = [search_tool, get_stock_price]"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Step 4: Initialize the LLM and Build the Agent"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# <<< MODIFIED: Initialize the DeepSeek model\n",
    "llm = ChatDeepseek(\n",
    "    model=\"deepseek-chat\", \n",
    "    temperature=0,\n",
    "    api_key=DEEPSEEK_API_KEY\n",
    ")\n",
    "\n",
    "# Create the prompt template. This tells the agent how to behave.\n",
    "prompt = ChatPromptTemplate.from_messages(\n",
    "    [\n",
    "        (\"system\", \"You are a helpful assistant.\"),\n",
    "        (\"placeholder\", \"{chat_history}\"),\n",
    "        (\"human\", \"{input}\"),\n",
    "        (\"placeholder\", \"{agent_scratchpad}\"),\n",
    "    ]\n",
    ")\n",
    "\n",
    "# Create the agent by binding the LLM with the tools and prompt\n",
    "agent = create_tool_calling_agent(llm, tools, prompt)\n",
    "\n",
    "# Create the Agent Executor, which runs the agent's reasoning loop\n",
    "agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Step 5: Run the Agent!\n",
    "\n",
    "Now we can ask the agent questions. The `verbose=True` setting will show the agent's thought process."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "#### Example 1: Using the Stock Price Tool"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "response1 = agent_executor.invoke({\"input\": \"What is the stock price of Apple (AAPL)?\"})"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\nFinal Answer:\", response1['output'])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "#### Example 2: Using Both Search and Stock Price Tools"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "response2 = agent_executor.invoke({\"input\": \"What was the latest news about NVIDIA and what is their current stock price (NVDA)?\"})"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\nFinal Answer:\", response2['output'])"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

NameError: name 'null' is not defined